<a href="https://colab.research.google.com/github/juancarloszuletacorcho-ops/Juan.Zuleta/blob/main/caso_practico_no_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Instalar las librerías requeridas para la  anipulación de datos, procesamiento del lenguaje natural, representación vectorial, visualización y construcción del modelo de aprendizaje profundo.###

In [ ]:
!pip install pandas numpy nltk scikit-learn gensim sentence-transformers plotly torch torchvision torchaudio -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 26.1 MB/s eta 0:00:00


### Cargue de base de datos ###


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving amazon_reviews.csv to amazon_reviews.csv
Saving IMDB Dataset.csv to IMDB Dataset.csv


In [ ]:
### Importar librerías ####
import pandas as pd
import numpy as np

import nltk
import re

from sklearn.model_selection import train_test_split

nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
import os

print(os.listdir())

['.config', 'amazon_reviews.csv', 'IMDB Dataset.csv', 'sample_data']


In [ ]:
amazon_df = pd.read_csv(
    "amazon_reviews.csv",
    low_memory=False
)

amazon_df.head()

,Unnamed: 0,reviewerName,overall,reviewText,reviewTime,day_diff,helpful_yes,helpful_no,total_vote,score_pos_neg_diff,score_average_rating,wilson_lower_bound
0,0,NaN,4.0,No issues.,2014-07-23,138,0,0,0,0,0.0,0.0
1,1,0mie,5.0,"Purchased this for my device, it worked as adv...",2013-10-25,409,0,0,0,0,0.0,0.0
2,2,1K3,4.0,it works as expected. I should have sprung for...,2012-12-23,715,0,0,0,0,0.0,0.0
3,3,1m2,5.0,This think has worked out great.Had a diff. br...,2013-11-21,382,0,0,0,0,0.0,0.0
4,4,2&amp;1/2Men,5.0,"Bought it with Retail Packaging, arrived legit...",2013-07-13,513,0,0,0,0,0.0,0.0


In [ ]:
print(amazon_df.shape)

(4915, 12)


In [ ]:
amazon_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4915 entries, 0 to 4914
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Unnamed: 0            4915 non-null   int64  
 1   reviewerName          4914 non-null   object 
 2   overall               4915 non-null   float64
 3   reviewText            4914 non-null   object 
 4   reviewTime            4915 non-null   object 
 5   day_diff              4915 non-null   int64  
 6   helpful_yes           4915 non-null   int64  
 7   helpful_no            4915 non-null   int64  
 8   total_vote            4915 non-null   int64  
 9   score_pos_neg_diff    4915 non-null   int64  
 10  score_average_rating  4915 non-null   float64
 11  wilson_lower_bound    4915 non-null   float64
dtypes: float64(3), int64(6), object(3)
memory usage: 460.9+ KB


In [ ]:
### se realiza la exploracion de los datos con el proposito de identificar su estructura de variables y verificar valores faltantes ademas de comprender las caracteristicas generales##

amazon_df.isnull().sum()

,0
Unnamed: 0,0
reviewerName,1
overall,0
reviewText,1
reviewTime,0
day_diff,0
helpful_yes,0
helpful_no,0
total_vote,0
score_pos_neg_diff,0


In [ ]:
### PROCESAMIENTO --- Seleccion de columnas para la eliminacion de columnas ###

amazon_df.columns


Index(['Unnamed: 0', 'reviewerName', 'overall', 'reviewText', 'reviewTime',
       'day_diff', 'helpful_yes', 'helpful_no', 'total_vote',
       'score_pos_neg_diff', 'score_average_rating', 'wilson_lower_bound'],
      dtype='object')

In [ ]:

amazon_df = amazon_df.dropna(
    subset=["reviewText"]
)

In [ ]:
###LIMPIEZA DE LOS DATOS ###

def clean_text(text):

    text = str(text)

    text = text.lower()

    text = re.sub(r"<.*?>"," ",text)

    text = re.sub(r"[^a-zA-Z ]"," ",text)

    text = re.sub(r"\s+"," ",text)

    return text.strip()

In [ ]:
amazon_df["clean_review"] = (
    amazon_df["reviewText"]
    .apply(clean_text)
)

In [ ]:
##### PARA reducir ruido, eliminar caracteres irrelevantes y normalizar el texto para mejorar la calidad de las representaciones vectoriales posteriores.###

amazon_df[
[
"reviewText",
"clean_review"
]
].head()

,reviewText,clean_review
0,No issues.,no issues
1,"Purchased this for my device, it worked as adv...",purchased this for my device it worked as adve...
2,it works as expected. I should have sprung for...,it works as expected i should have sprung for ...
3,This think has worked out great.Had a diff. br...,this think has worked out great had a diff bra...
4,"Bought it with Retail Packaging, arrived legit...",bought it with retail packaging arrived legit ...


In [ ]:
## BOW ###
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer(
    max_features=1000
)

X_bow = bow.fit_transform(
    amazon_df["clean_review"]
)

In [ ]:
### Cada fila representa una reseña y cada columna representa una palabra del vocabulario. ####

X_bow.shape

(4914, 1000)

In [ ]:
### TF-IDF  #

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=1000
)

X_tfidf = tfidf.fit_transform(
    amazon_df["clean_review"]
)

In [ ]:
### validacion ###

X_tfidf.shape

(4914, 1000)

In [ ]:
#### WORD2VEC  ###

from gensim.models import Word2Vec

sentences = [
    text.split()
    for text in amazon_df["clean_review"]
]

In [ ]:
w2v_model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

In [ ]:
#### ### validacion ###

w2v_model.wv.most_similar(
    "good"
)

[('excellent', 0.9149923324584961),
 ('reasonable', 0.9117640256881714),
 ('quality', 0.902877926826477),
 ('reliable', 0.8938032984733582),
 ('value', 0.8867697715759277),
 ('pleased', 0.8857660293579102),
 ('deal', 0.8740404844284058),
 ('very', 0.8724820017814636),
 ('disappoint', 0.8707131147384644),
 ('happy', 0.8650534749031067)]

In [ ]:
#### SENTENCE TRANSFORMERS ###

from sentence_transformers import SentenceTransformer

model_st = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
sample_reviews = amazon_df[
"clean_review"
].head(200)

embeddings = model_st.encode(
    sample_reviews.tolist()
)

#### ### validacion ###

embeddings.shape

(200, 384)

In [ ]:
### VISUALIZACIÓN PLOTLY ####

from sklearn.decomposition import PCA

pca = PCA(n_components=2)

emb_2d = pca.fit_transform(
    embeddings
)

In [ ]:
### VISUALIZACIÓN PLOTLY ####

from sklearn.decomposition import PCA

pca = PCA(n_components=2)

emb_2d = pca.fit_transform(
    embeddings
)

import plotly.express as px

fig = px.scatter(
    x=emb_2d[:,0],
    y=emb_2d[:,1]
)

fig.show()

In [ ]:
### LSTM IMDB ##### CARGA DEL DATASET IMDB   ####

## -- El conjunto de datos IMDB contiene 50.000 reseñas de películas previamente etiquetadas como positivas o negativas, constituyendo un problema clásico de clasificación binaria supervisada.--- ###

imdb_df = pd.read_csv(
    "IMDB Dataset.csv"
)

print(imdb_df.shape)

imdb_df.head()

(50000, 2)


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
### INSPECCIÓN Y VALIDACIÓN ###

imdb_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [ ]:
imdb_df = imdb_df.dropna()

Antes de entrenar cualquier modelo es necesario verificar la calidad de los datos, identificando posibles valores faltantes que puedan afectar el rendimiento del algoritmo

In [ ]:
###  PREPROCESAMIENTO ####

import re

In [ ]:
def clean_text(text):

    text = str(text)

    text = text.lower()

    text = re.sub(r'<.*?>', ' ', text)

    text = re.sub(r'[^a-zA-Z ]', ' ', text)

    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [ ]:
imdb_df["clean_review"] = (
    imdb_df["review"]
    .apply(clean_text)
)

In [ ]:
### limpieza textual permite para eliminar ruido, normalizar la información y mejorar la calidad de las representaciones vectoriales generadas posteriormente.###

imdb_df[
[
"review",
"clean_review"
]
].head()

,review,clean_review
0,One of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...
1,A wonderful little production. <br /><br />The...,a wonderful little production the filming tech...
2,I thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...
3,Basically there's a family where a little boy ...,basically there s a family where a little boy ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei s love in the time of money is a...


In [ ]:
### CONVERSIÓN DE ETIQUETAS ###

imdb_df["label"] = imdb_df[
    "sentiment"
].map(
{
    "positive":1,
    "negative":0
}
)

In [ ]:
#### Se transformaron las etiquetas categóricas en variables numéricas binarias, donde 1 representa sentimiento positivo y 0 sentimiento negativo. ###

imdb_df["label"].value_counts()

,count
label,
1,25000
0,25000


In [ ]:
### TOKENIZACIÓN ###

from collections import Counter

all_words = []

for review in imdb_df["clean_review"]:

    all_words.extend(
        review.split()
    )

In [ ]:
all_words = []

for review in imdb_df["clean_review"]:

    all_words.extend(
        review.split()
    )

In [ ]:
counter = Counter(
    all_words
)

In [ ]:
print(counter.most_common(10))
print(f"Total unique words in counter: {len(counter)}")

NameError: name 'counter' is not defined

In [ ]:
import pandas as pd
from collections import Counter
import re

# 1. Cargar el dataset si no existe
try:
    imdb_df = pd.read_csv('IMDB Dataset.csv')
    print("Dataset cargado correctamente.")
except FileNotFoundError:
    print("Error: No se encontró el archivo 'IMDB Dataset.csv'. Asegúrate de haberlo subido.")

# 2. Limpieza mínima necesaria
def quick_clean(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'[^a-zA-Z ]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

imdb_df['clean_review'] = imdb_df['review'].apply(quick_clean)

# 3. Inicializar all_words
all_words = []
for review in imdb_df['clean_review']:
    all_words.extend(review.split())

# 4. Crear counter y vocab
counter = Counter(all_words)
vocab = {word: i+1 for i, (word, count) in enumerate(counter.items())}

print(f"Vocabulario creado con {len(vocab)} palabras.")
print("Top 5 palabras:", counter.most_common(5))

Dataset cargado correctamente.
Vocabulario creado con 99420 palabras.
Top 5 palabras: [('the', 668009), ('and', 324439), ('a', 323060), ('of', 289415), ('to', 268123)]


In [ ]:
vocab = {
word:i+1
for i,(word,count)
in enumerate(
counter.items()
)
}
#### ### validacion ###
len(vocab)

99420

In [ ]:
### ENCODING ###

MAX_LEN = 200

In [ ]:
def encode(text):

    encoded = [

        vocab.get(word,0)

        for word in text.split()

    ]

    encoded = encoded[:MAX_LEN]

    if len(encoded) < MAX_LEN:

        encoded += [0] * (
            MAX_LEN - len(encoded)
        )

    return encoded

In [ ]:
y = np.array(
    imdb_df["label"]
)

In [ ]:
### TRAIN TEST SPLIT ###

from sklearn.model_selection import train_test_split

In [ ]:
import numpy as np
import torch
from sklearn.model_selection import train_test_split

# 1. Asegurar constantes y función de encoding
MAX_LEN = 200

def encode(text):
    encoded = [vocab.get(word, 0) for word in str(text).split()]
    encoded = encoded[:MAX_LEN]
    if len(encoded) < MAX_LEN:
        encoded += [0] * (MAX_LEN - len(encoded))
    return encoded

print("Preparando tensores de entrenamiento y prueba...")

# 2. Asegurar que la columna 'label' exista
if 'label' not in imdb_df.columns:
    imdb_df['label'] = imdb_df['sentiment'].map({'positive': 1, 'negative': 0})

# 3. Aplicar encoding y preparar etiquetas
X = np.array(imdb_df['clean_review'].apply(encode).tolist())
y = np.array(imdb_df['label'].tolist())

# 4. Dividir el dataset (80% train, 20% test)
X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 5. Convertir a Tensores de PyTorch
X_train = torch.tensor(X_train_np, dtype=torch.long)
X_test = torch.tensor(X_test_np, dtype=torch.long)
y_train = torch.tensor(y_train_np, dtype=torch.float32)
y_test = torch.tensor(y_test_np, dtype=torch.float32)

print(f"Preparación completada:")
print(f"- X_train: {X_train.shape}")
print(f"- X_test: {X_test.shape}")

Preparando tensores de entrenamiento y prueba...
Preparación completada:
- X_train: torch.Size([40000, 200])
- X_test: torch.Size([10000, 200])


In [ ]:
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_test: (10000, 200)
Shape of y_train: (40000,)
Shape of y_test: (10000,)


In [ ]:
X_train.shape

(40000, 200)

In [ ]:
### CONVERTIR A TENSORES ##

import torch

X_train = torch.tensor(
    X_train,
    dtype=torch.long
)

X_test = torch.tensor(
    X_test,
    dtype=torch.long
)

y_train = torch.tensor(
    y_train,
    dtype=torch.float32
)

y_test = torch.tensor(
    y_test,
    dtype=torch.float32
)

In [ ]:
### GPU  #

device = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"
)

In [ ]:
print(device)


cpu


In [ ]:
### CONSTRUCCIÓN DE LA LSTM ###

import torch.nn as nn

class SentimentLSTM(
    nn.Module
):

    def __init__(
        self,
        vocab_size
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size+1,
            128
        )

        self.lstm = nn.LSTM(
            128,
            128,
            batch_first=True
        )

        self.dropout = nn.Dropout(
            0.3
        )

        self.fc = nn.Linear(
            128,
            1
        )

        self.sigmoid = nn.Sigmoid()

    def forward(
        self,
        x
    ):

        x = self.embedding(x)

        _,(hidden,_) = self.lstm(x)

        x = self.dropout(
            hidden[-1]
        )

        x = self.fc(x)

        return self.sigmoid(x)

Las redes LSTM permiten capturar dependencias secuenciales dentro del lenguaje, resultando especialmente útiles para tareas de clasificación de sentimientos.

In [ ]:
# model = SentimentLSTM(
#     len(vocab)
# ).to(device)
# The model instantiation has been moved to the training setup cell for robustness.

In [ ]:
import torch
import torch.nn as nn

# 1. Definir el dispositivo (GPU o CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

# 2. Verificar que 'vocab' existe (si no, lo tomamos del estado actual)
try:
    vocab_size = len(vocab)
except NameError:
    # Re-crear vocab si accidentalmente se perdió el estado
    vocab = {word: i+1 for i, (word, count) in enumerate(counter.items())}
    vocab_size = len(vocab)

# 3. Instanciar el modelo
model = SentimentLSTM(vocab_size).to(device)

# 4. Configurar pérdida y optimizador
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Modelo inicializado correctamente y listo para entrenar.")

Usando dispositivo: cpu
Modelo inicializado correctamente y listo para entrenar.


In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

# 1. Recuperación de emergencia: si X_train no existe, lo recreamos
try:
    _ = X_train
except NameError:
    print("Tensores no encontrados. Regenerando datos...")
    MAX_LEN = 200
    def encode(text):
        encoded = [vocab.get(word, 0) for word in str(text).split()]
        encoded = encoded[:MAX_LEN]
        if len(encoded) < MAX_LEN: encoded += [0] * (MAX_LEN - len(encoded))
        return encoded

    if 'label' not in imdb_df.columns:
        imdb_df['label'] = imdb_df['sentiment'].map({'positive': 1, 'negative': 0})

    X_all = np.array(imdb_df['clean_review'].apply(encode).tolist())
    y_all = np.array(imdb_df['label'].tolist())

    from sklearn.model_selection import train_test_split
    X_tr_np, X_te_np, y_tr_np, y_te_np = train_test_split(X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)

    X_train = torch.tensor(X_tr_np, dtype=torch.long)
    y_train = torch.tensor(y_tr_np, dtype=torch.float32)
    X_test = torch.tensor(X_te_np, dtype=torch.long)
    y_test = torch.tensor(y_te_np, dtype=torch.float32)

# 2. Configurar DataLoader
batch_size = 64
train_data = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_data, shuffle=True, batch_size=batch_size)

# 3. Entrenamiento
model.to(device)
EPOCHS = 3
print(f"Entrenando en {device}...")

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    for batch_idx, (batch_X, batch_y) in enumerate(train_loader):
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_X).squeeze()
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        if (batch_idx + 1) % 100 == 0:
            print(f"Epoch {epoch+1}/{EPOCHS} | Batch {batch_idx+1}/{len(train_loader)} | Loss: {loss.item():.4f}")
    print(f"--- Epoch {epoch+1} Finalizada. Promedio Loss: {epoch_loss/len(train_loader):.4f} ---")

print("¡Entrenamiento completado!")

NameError: name 'X_train' is not defined

In [ ]:
### EVALUACIÓN ###

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [ ]:
model.eval()

with torch.no_grad():

    preds = model(

        X_test.to(device)

    ).squeeze()

NameError: name 'model' is not defined

### Corrected Model Setup, Training, and Evaluation ###

It appears that `NameError`s related to `vocab` and `model` have occurred. This typically means that the cells defining these variables were either not executed in the correct order or the kernel state was reset.

To ensure all necessary components are correctly initialized, please run the following cells sequentially. This will redefine `vocab` (if necessary), instantiate the `SentimentLSTM` model, set up the training parameters, run the training loop, and then perform the evaluation.

In [ ]:
# Ensure vocab is defined (copied from eBUWizNLBP25 for robustness)
vocab = {
    word:i+1
    for i,(word,count)
    in enumerate(
        counter.items()
    )
}
print(f"Length of vocab: {len(vocab)}")

NameError: name 'counter' is not defined

In [ ]:
# Model Instantiation and Training Setup (corrected from 7_udB0GbGCBL)
model = SentimentLSTM(
    len(vocab)
).to(device)

criterion = nn.BCELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Model, criterion, and optimizer have been successfully set up.")

NameError: name 'vocab' is not defined

In [ ]:
# Training Loop (copied from XsIDZZqkGPkv)
EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()

    outputs = model(
        X_train.to(device)
    ).squeeze()

    loss = criterion(
        outputs,
        y_train.to(device)
    )

    loss.backward()
    optimizer.step()

    print(
        f"Epoch {epoch+1}"
        f" Loss={loss.item():.4f}"
    )

NameError: name 'model' is not defined

In [ ]:
# Evaluation Setup and Execution (copied from FpGYnq0jG-IQ and b6LX9O-FH4Jm)
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

model.eval()

with torch.no_grad():
    preds = model(
        X_test.to(device)
    ).squeeze()

    # Convert predictions to binary (0 or 1)
    preds_binary = (preds >= 0.5).long()

    # Convert y_test to CPU and long for comparison
    y_test_cpu = y_test.cpu().long()

    accuracy = accuracy_score(y_test_cpu, preds_binary)
    report = classification_report(y_test_cpu, preds_binary)
    conf_matrix = confusion_matrix(y_test_cpu, preds_binary)

print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:")
print(report)
print("Confusion Matrix:")
print(conf_matrix)


NameError: name 'model' is not defined